In [1]:
import myelin_extraction
import imageryclient as ic
import pandas as pd
from caveclient import CAVEclient
from classifiers import SimpleClassifier
import pickle

#for debugging
import importlib

In [2]:
client = CAVEclient('minnie65_public')
# client.materialize.get_versions()
client.version = 1507 #1507, 1412...

img_client = ic.ImageryClient(client=client)

In [3]:
neurons_df = client.materialize.query_table("allen_v1_column_types_slanted_ref")
cell_type_df = client.materialize.query_table('aibs_metamodel_celltypes_v661')
merged_df = pd.merge(neurons_df, cell_type_df, on='pt_root_id', how='inner')
proof_df = client.materialize.tables.proofreading_status_and_strategy(status_axon="t").query(
    select_columns=['pt_root_id','status_axon','status_dendrite','strategy_axon','strategy_dendrite'],
)
merged_df = pd.merge(merged_df, proof_df, on='pt_root_id', how='inner')
final_df = merged_df[['pt_root_id', 'cell_type_x', 'strategy_axon', 'strategy_dendrite']]

ax_full_extend_df = final_df.query("strategy_axon == 'axon_fully_extended'")

#remove all entries from ax_full_extend_df where cell_type_x = 'BC'ArithmeticError
ax_full_extend_df = ax_full_extend_df.query("cell_type_x != 'BC'")

full_extend_ids = ax_full_extend_df["pt_root_id"].to_list()

ax_full_extend_df.iloc[0:40]



,pt_root_id,cell_type_x,strategy_axon,strategy_dendrite
10,864691135741655956,23P,axon_fully_extended,dendrite_extended
36,864691136674080135,23P,axon_fully_extended,dendrite_extended
55,864691136110765496,MC,axon_fully_extended,dendrite_extended
63,864691135256861871,5P-IT,axon_fully_extended,dendrite_extended
66,864691136330122730,5P-IT,axon_fully_extended,dendrite_extended
83,864691135878061395,5P-IT,axon_fully_extended,dendrite_extended
139,864691135258083503,23P,axon_fully_extended,dendrite_clean
169,864691135890167433,5P-IT,axon_fully_extended,dendrite_extended
184,864691136974813212,5P-IT,axon_fully_extended,dendrite_extended
190,864691136445215875,23P,axon_fully_extended,dendrite_extended


In [4]:
all_full_extend_df = proof_df.query("strategy_axon == 'axon_fully_extended'")

#merge all_full_extend_df with cell_type_df on pt_root_id, left
all_full_extend_df = pd.merge(all_full_extend_df, cell_type_df, on='pt_root_id', how='left')

# all_full_extend_df = all_full_extend_df.query("cell_type != 'BC'") #remove all BC



all_full_extend_ids = all_full_extend_df["pt_root_id"].to_list()

# all_full_extend_df.iloc[0:40]

print(len(all_full_extend_ids))

# print(all_full_extend_ids[49:-1])

267


In [26]:
all_full_extend_df.iloc[200:250]

,pt_root_id,status_axon,status_dendrite,strategy_axon,strategy_dendrite,id_ref,created_ref,valid_ref,volume,pt_supervoxel_id,id,created,valid,target_id,classification_system,cell_type,pt_position,bb_start_position,bb_end_position
200,864691136453186559,t,t,axon_fully_extended,dendrite_extended,303115.0,2020-09-28 22:43:31.751462+00:00,t,356.344955,9.170923e+16,31288.0,2023-12-19 22:45:52.183540+00:00,t,303115.0,inhibitory_neuron,BC,"[195696, 200832, 21084]","[nan, nan, nan]","[nan, nan, nan]"
201,864691135442591048,t,t,axon_fully_extended,dendrite_extended,269557.0,2020-09-28 22:44:00.395171+00:00,t,201.339863,8.770021e+16,25915.0,2023-12-19 22:44:29.694326+00:00,t,269557.0,inhibitory_neuron,BC,"[166608, 215680, 23067]","[nan, nan, nan]","[nan, nan, nan]"
202,864691135989392387,t,t,axon_fully_extended,dendrite_extended,330372.0,2020-09-28 22:43:39.857212+00:00,t,365.302743,9.205551e+16,35325.0,2023-12-19 22:46:54.312499+00:00,t,330372.0,inhibitory_neuron,BC,"[198624, 159104, 21248]","[nan, nan, nan]","[nan, nan, nan]"
203,864691136011458734,t,t,axon_fully_extended,dendrite_extended,260575.0,2020-09-28 22:43:48.879799+00:00,t,365.960233,8.748154e+16,23512.0,2023-12-19 22:43:53.619432+00:00,t,260575.0,inhibitory_neuron,BC,"[164944, 159008, 21409]","[nan, nan, nan]","[nan, nan, nan]"
204,864691136274984638,t,t,axon_fully_extended,dendrite_extended,260824.0,2020-09-28 22:40:53.035083+00:00,t,323.171779,8.881834e+16,23624.0,2023-12-19 22:43:55.001389+00:00,t,260824.0,inhibitory_neuron,BC,"[174688, 157696, 23299]","[nan, nan, nan]","[nan, nan, nan]"
205,864691135760685390,t,t,axon_fully_extended,dendrite_extended,260672.0,2020-09-28 22:43:32.206822+00:00,t,347.295908,8.755129e+16,23557.0,2023-12-19 22:43:54.162211+00:00,t,260672.0,inhibitory_neuron,BC,"[165744, 154416, 22523]","[nan, nan, nan]","[nan, nan, nan]"
206,864691135702846075,t,t,axon_fully_extended,dendrite_extended,330650.0,2020-09-28 22:45:07.421935+00:00,t,321.153925,9.226559e+16,35411.0,2023-12-19 22:46:55.496291+00:00,t,330650.0,inhibitory_neuron,BC,"[199920, 151248, 23050]","[nan, nan, nan]","[nan, nan, nan]"
207,864691135106209101,t,t,axon_fully_extended,dendrite_extended,260206.0,2020-09-28 22:44:41.831013+00:00,t,274.336317,8.895778e+16,23419.0,2023-12-19 22:43:52.493949+00:00,t,260206.0,inhibitory_neuron,BC,"[175744, 148128, 19338]","[nan, nan, nan]","[nan, nan, nan]"
208,864691135388983425,t,t,axon_fully_extended,dendrite_extended,296421.0,2020-09-28 22:45:12.026726+00:00,t,337.884447,9.106884e+16,29346.0,2023-12-19 22:45:23.440856+00:00,t,296421.0,inhibitory_neuron,BC,"[191024, 147792, 19680]","[nan, nan, nan]","[nan, nan, nan]"
209,864691135499624723,t,t,axon_fully_extended,dendrite_extended,296378.0,2020-09-28 22:45:15.525967+00:00,t,349.033103,8.987243e+16,29326.0,2023-12-19 22:45:23.194192+00:00,t,296378.0,inhibitory_neuron,BC,"[182784, 146896, 19849]","[nan, nan, nan]","[nan, nan, nan]"


In [8]:
ax_full_extend_df.iloc[45:-1]

,pt_root_id,cell_type_x,strategy_axon,strategy_dendrite
778,864691135655390658,23P,axon_fully_extended,dendrite_extended
808,864691135920571056,4P,axon_fully_extended,dendrite_extended
810,864691135701676411,23P,axon_fully_extended,dendrite_extended
832,864691136144674612,23P,axon_fully_extended,dendrite_extended
839,864691135646493679,5P-IT,axon_fully_extended,dendrite_extended
849,864691135350542039,5P-IT,axon_fully_extended,dendrite_extended
866,864691135346975135,5P-IT,axon_fully_extended,dendrite_extended
871,864691134886404090,5P-IT,axon_fully_extended,dendrite_extended
936,864691135891467401,5P-IT,axon_fully_extended,dendrite_extended
962,864691135325071004,23P,axon_fully_extended,dendrite_extended


In [7]:
importlib.reload(myelin_extraction) #delete this later....

classifier_model = SimpleClassifier()
model_weights_file = "DNN_classifiers/myelin_classifier_v0_1.pt"
max_workers=23 #i like 23
box_sz_microns = "adaptive"
length_thresh=3000 #3000 is default.

#72...

#115

#go back to 105....

myelin_extraction.process_neurons(all_full_extend_ids[137:-1], client, img_client, classifier_model, model_weights_file, max_workers, box_sz_microns, length_thresh)

# myelin_extraction.process_neurons([864691136033871035], client, img_client, classifier_model, model_weights_file, max_workers, box_sz_microns, length_thresh)


File segments_myelin_1507/864691135375430985.pkl already exists. Skipping neuron 864691135375430985.
File segments_myelin_1507/864691135927533070.pkl already exists. Skipping neuron 864691135927533070.
File segments_myelin_1507/864691136086206316.pkl already exists. Skipping neuron 864691136086206316.
File segments_myelin_1507/864691135196040746.pkl already exists. Skipping neuron 864691135196040746.
File segments_myelin_1507/864691136363828194.pkl already exists. Skipping neuron 864691136363828194.
File segments_myelin_1507/864691135954021923.pkl already exists. Skipping neuron 864691135954021923.
File segments_myelin_1507/864691135988769027.pkl already exists. Skipping neuron 864691135988769027.
File segments_myelin_1507/864691135572576237.pkl already exists. Skipping neuron 864691135572576237.
File segments_myelin_1507/864691136445215875.pkl already exists. Skipping neuron 864691136445215875.
File segments_myelin_1507/864691135464263742.pkl already exists. Skipping neuron 8646911354

KeyboardInterrupt: 

In [19]:
#Extracting partially extended neurons.

ax_part_extend_df = proof_df.query("strategy_axon == 'axon_partially_extended'")
ax_part_extend_pt_root_ids = ax_part_extend_df['pt_root_id'].values

#filter for cell type == 4P
ax_part_extend_df = pd.merge(ax_part_extend_df, cell_type_df, on='pt_root_id', how='left')
ax_part_extend_df = ax_part_extend_df.query("cell_type == '4P'")
ax_part_extend_pt_root_ids = ax_part_extend_df['pt_root_id'].values

classifier_model = SimpleClassifier()
model_weights_file = "DNN_classifiers/myelin_classifier_v0_1.pt"
max_workers=17 #i like 23
box_sz_microns = "adaptive"
length_thresh=3000 #3000 is default.


myelin_extraction.process_neurons(ax_part_extend_pt_root_ids[15:-1], client, img_client, classifier_model, model_weights_file, max_workers, box_sz_microns, length_thresh)


File segments_myelin_1507/864691135742189163.pkl already exists. Skipping neuron 864691135742189163.
File segments_myelin_1507/864691136025333561.pkl already exists. Skipping neuron 864691136025333561.
Processing neuron 864691135101521952
Starting segment 1/45
Processed point: [151962 204316  23561]: Myelin=0.0
Processed point: [154426 202984  23442]: Myelin=0.0
Processed point: [154008 203096  23477]: Myelin=0.0
Processed point: [154594 202950  23426]: Myelin=0.0
Processed point: [154926 202760  23415]: Myelin=0.0
Processed point: [150758 205000  23560]: Myelin=0.0
Processed point: [152364 204056  23553]: Myelin=0.0
Processed point: [149012 205850  23626]: Myelin=0.0
Processed point: [150106 205534  23573]: Myelin=0.0
Processed point: [153168 203518  23540]: Myelin=0.0
Processed point: [155202 202504  23403]: Myelin=0.0
Processed point: [151792 204698  23565]: Myelin=0.0
Processed point: [150854 204950  23559]: Myelin=0.0
Processed point: [150372 205420  23563]: Myelin=0.0
Processed p

KeyboardInterrupt: 

In [10]:
#Extracting partially extended neurons with a focus on the ones projecting to hva.

with open('ax_partially_extended_but_interareal.pkl', 'rb') as f:
    pt_root_ids = pickle.load(f)

classifier_model = SimpleClassifier()
model_weights_file = "DNN_classifiers/myelin_classifier_v0_1.pt"
max_workers=25 #i like 23
box_sz_microns = "adaptive"
length_thresh=3000 #3000 is default.

# reverse pt_root_ids list, so it starts from the end.
pt_root_ids = pt_root_ids[::-1]


myelin_extraction.process_neurons(pt_root_ids[40:-1], client, img_client, classifier_model, model_weights_file, max_workers, box_sz_microns, length_thresh)


File segments_myelin_1507/864691135469924178.pkl already exists. Skipping neuron 864691135469924178.
File segments_myelin_1507/864691136380268245.pkl already exists. Skipping neuron 864691136380268245.
File segments_myelin_1507/864691136380795861.pkl already exists. Skipping neuron 864691136380795861.
File segments_myelin_1507/864691135778465376.pkl already exists. Skipping neuron 864691135778465376.
File segments_myelin_1507/864691135309141062.pkl already exists. Skipping neuron 864691135309141062.
File segments_myelin_1507/864691136135997579.pkl already exists. Skipping neuron 864691136135997579.
File segments_myelin_1507/864691135725838763.pkl already exists. Skipping neuron 864691135725838763.
File segments_myelin_1507/864691136380816597.pkl already exists. Skipping neuron 864691136380816597.
File segments_myelin_1507/864691135214653240.pkl already exists. Skipping neuron 864691135214653240.
File segments_myelin_1507/864691135209715577.pkl already exists. Skipping neuron 8646911352

KeyboardInterrupt: 

864691135476495936 - i like this neuron.

caffeinate -i -t 259200